# Autopsie SRPG Prooftag — étape par étape

Ce notebook sépare ce que le modèle génère de ce que la réparation déterministe ajoute. Il ne considère jamais `final.png` comme une preuve de réussite du modèle.

In [ ]:
from pathlib import Path
from IPython.display import Markdown, display
from PIL import Image
import csv, json, os, tarfile
import matplotlib.pyplot as plt
import numpy as np

CASE = 'botanical-short'
ARCHIVE = os.environ.get('PROOFTAG_QR_BENCHMARK_ARCHIVE')
if ARCHIVE is None:
    roots = [Path.home() / 'Downloads' / 'prooftag-benchmarks', Path.cwd() / 'benchmark-results']
    candidates = sorted((p for root in roots if root.exists() for p in root.glob('*.tar.gz')), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError('Définir PROOFTAG_QR_BENCHMARK_ARCHIVE vers une archive .tar.gz')
    archive_path = candidates[-1]
else:
    archive_path = Path(ARCHIVE).expanduser().resolve()
archive_path

In [ ]:
run_name = archive_path.name.removesuffix('.tar.gz')
extracted_dir = archive_path.parent / run_name
summary_path = extracted_dir / 'summary.json'
if not summary_path.exists():
    cache_root = Path(os.environ.get('PROOFTAG_QR_NOTEBOOK_CACHE', archive_path.parent / '.prooftag-notebook-cache'))
    cache_root.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path, 'r:gz') as bundle:
        root = cache_root.resolve()
        for member in bundle.getmembers():
            target = (cache_root / member.name).resolve()
            if root not in target.parents and target != root:
                raise ValueError(f'Chemin dangereux dans l’archive: {member.name}')
        bundle.extractall(cache_root)
    extracted_dir = cache_root / run_name
    summary_path = extracted_dir / 'summary.json'
run_dir = summary_path.parent
summary = json.loads(summary_path.read_text(encoding='utf-8'))
environment = json.loads((run_dir / 'environment.json').read_text(encoding='utf-8'))
case_dir = run_dir / 'cases' / CASE
result = next(row for row in summary['results'] if row['case'] == CASE)
display(Markdown(f"**Archive :** `{archive_path.name}`  \n**Commit :** `{summary['git_commit']}`  \n**Cas :** `{CASE}`  \n**Profil final :** `{result.get('selected_variant')}`"))

## 1. Ce qui est réellement livré
`raw` est la première diffusion, `srpg` est la deuxième diffusion guidée sans réparation, et `final` est la variante choisie après tous les fallbacks.

In [ ]:
stages = [('Brut du modèle', case_dir / 'raw.png'), ('SRPG sans réparation', case_dir / 'srpg.png'), ('Final livré', case_dir / 'final.png')]
available = [(title, path) for title, path in stages if path.exists()]
fig, axes = plt.subplots(1, len(available), figsize=(6 * len(available), 6))
axes = np.atleast_1d(axes)
for ax, (title, path) in zip(axes, available):
    ax.imshow(Image.open(path).convert('RGB'))
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()

In [ ]:
display(Markdown(
    f"**Lecture brute :** {result.get('raw_scan_pass_rate')}  \n"
    f"**Lecture SRPG :** {result.get('srpg_scan_pass_rate')}  \n"
    f"**État interne SRPG :** `{result.get('srpg_stage_status')}`  \n"
    f"**Pixels modifiés dans le final :** {100 * (result.get('changed_pixel_ratio') or 0):.1f}%"
))

## 2. Les 40 pas de guidage
Une baisse de la loss interne n'est pas une preuve de scan. On affiche ensemble erreur centrale, SRL, LPIPS et amplitude du guidage.

In [ ]:
with (run_dir / 'srpg-steps.csv').open(encoding='utf-8', newline='') as stream:
    steps = [row for row in csv.DictReader(stream) if row['case'] == CASE and row['attempt'] == '1']
x = [int(row['index']) for row in steps]
fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
series = [('module_error_rate', 'Erreur centrale'), ('srl', 'SRL'), ('lpips', 'LPIPS'), ('noise_delta_rms', 'Delta bruit RMS')]
for ax, (field, title) in zip(axes.flat, series):
    ax.plot(x, [float(row[field]) for row in steps], marker='o', markersize=3)
    ax.set_title(title); ax.grid(alpha=.25); ax.set_xlabel('Étape DDIM')
plt.tight_layout()

## 3. Estimations propres `x0` pendant la diffusion
Ces images sont exactement celles décodées pour calculer SRL + LPIPS. Une ancienne archive peut ne pas encore les contenir.

In [ ]:
x0_paths = sorted(case_dir.glob('attempt_1_srpg_step_*_x0.png'))
if not x0_paths:
    display(Markdown('⚠️ **Archive antérieure à la capture visuelle.** Redéployer puis relancer `make benchmark-e005`.'))
else:
    columns = 4; rows = int(np.ceil(len(x0_paths) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(16, 4 * rows))
    for ax in np.asarray(axes).reshape(-1): ax.axis('off')
    for ax, path in zip(np.asarray(axes).reshape(-1), x0_paths):
        ax.imshow(Image.open(path).convert('RGB')); ax.set_title(path.stem.replace('attempt_1_', '')); ax.axis('off')
    plt.tight_layout()

## 4. Modules encore faux aux mêmes étapes
Blanc = module central encore incorrect. Cette carte permet de vérifier si le modèle corrige la structure ou se contente de faire baisser un surrogate instable.

In [ ]:
error_paths = sorted(case_dir.glob('attempt_1_srpg_step_*_errors.png'))
if error_paths:
    columns = 4; rows = int(np.ceil(len(error_paths) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(16, 4 * rows))
    for ax in np.asarray(axes).reshape(-1): ax.axis('off')
    for ax, path in zip(np.asarray(axes).reshape(-1), error_paths):
        ax.imshow(Image.open(path), cmap='magma', vmin=0, vmax=255); ax.set_title(path.stem.replace('attempt_1_', '')); ax.axis('off')
    plt.tight_layout()

## 5. Où les modules visibles sont ajoutés
La rangée suivante affiche les réparations intermédiaires disponibles. Le nom du profil final permet d'identifier la vraie source du rendu livré.

In [ ]:
variant_paths = sorted(p for p in case_dir.glob('attempt_1_*.png') if 'step_' not in p.name and p.name not in {'attempt_1_raw.png', 'attempt_1_srpg.png'})
if variant_paths:
    columns = 4; rows = int(np.ceil(len(variant_paths) / columns))
    fig, axes = plt.subplots(rows, columns, figsize=(16, 4 * rows))
    for ax in np.asarray(axes).reshape(-1): ax.axis('off')
    for ax, path in zip(np.asarray(axes).reshape(-1), variant_paths):
        ax.imshow(Image.open(path).convert('RGB')); ax.set_title(path.stem.replace('attempt_1_', '')); ax.axis('off')
    plt.tight_layout()

In [ ]:
if result.get('selected_variant', '').startswith(('perceptual_', 'rounded_', 'uncertain_', 'incorrect_', 'tonal_', 'centers_')):
    display(Markdown('## Conclusion automatique\nLe `final` ne vient **pas** de SRPG. SRPG a été abandonné et les modules visibles ont été ajoutés par la réparation déterministe sélectionnée. Il faut améliorer la diffusion avant de juger le rendu final.'))
else:
    display(Markdown('## Conclusion automatique\nLe profil final provient de la branche générative. Vérifier ses scans et son évolution visuelle avant toute promotion.'))